# Experiment 3: Stronger Training Recipe + Diverse Ensemble (ResNet50, ConvNeXt-Tiny, EfficientNetV2-S)

**Purpose.** Experiments 1-2 showed that the EfficientNetB0+ResNet50 fusion is statistically about equal to a fairly-tuned ResNet50 alone
(fusion 90.56% +/- 2.21; ResNet50-only 91.35% +/- 1.15) and that a 5-seed fusion ensemble reaches about 92%. This notebook asks:
**can a stronger, still-honest recipe and a more diverse ensemble do better?**

What changes compared with Experiments 1-2:
1. **Stronger backbones in the comparison:** ResNet50 (improved recipe), ConvNeXt-Tiny, EfficientNetV2-S.
2. **Better fine-tuning recipe:** every backbone layer except BatchNorm is unfrozen in Phase 2, AdamW + cosine learning-rate decay, label smoothing (0.1),
   stronger geometric augmentation (flip, small rotation/translation/zoom, contrast).
3. **Test-time augmentation (TTA):** predictions averaged over the image and its horizontal flip. Both plain and TTA numbers are reported.
4. **Ensembles:** per-architecture 5-seed ensembles and one diverse ensemble over all models.

**Honesty rules kept from before:** the same fixed stratified 70/15/15 split (SPLIT_SEED=42), same 5 training seeds, checkpoint selection on the **validation** set only,
the test set is used once per model for final evaluation, and every result is saved incrementally so nothing is lost if the session stops.

**Kaggle setup:** add the same dataset (896 / 64 / 3200 / 2240 fingerprint), GPU T4 x2 or P100, Internet ON (for ImageNet weights).
Then *Save Version -> Save & Run All (Commit)*.

**Version 2** - Phase-2 learning rate lowered to 2e-5 with gradient clipping after the first pilot collapsed at 1e-4.

**Estimated runtime:** roughly 25-40 min per seed per architecture group, i.e. about 3-5 hours for 5 seeds x 3 architectures. If you want a quick pilot first,
set `SEEDS = [42]` in cell 1 (about 40 min).

In [ ]:
# 1. Environment and configuration
import os, json, time, random, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)

DRY_RUN = os.environ.get("DRY_RUN") == "1"          # only used by the author for a CPU smoke test
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
WEIGHTS = None if DRY_RUN else "imagenet"
IMG_SIZE = (64, 64) if DRY_RUN else (224, 224)
BATCH_SIZE = 8 if DRY_RUN else 32
P1_EPOCHS = 1 if DRY_RUN else 8        # Phase 1: backbone frozen, head only
P2_EPOCHS = 1 if DRY_RUN else 30       # Phase 2: fine-tuning with cosine decay
SEEDS = [42] if DRY_RUN else [42, 123, 456, 789, 2024]
ARCHS = ["resnet50", "convnext_tiny", "efficientnetv2_s"]
LABEL_SMOOTHING = 0.1
SPLIT_SEED = 42

if not DRY_RUN:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")   # faster on T4/P100, softmax stays float32

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Seeds:", SEEDS, "| Architectures:", ARCHS)


In [ ]:
# 2. Locate dataset and build the file index (same logic as Experiments 1-2)
CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
label_to_idx = {c: i for i, c in enumerate(CLASSES)}

def normalize(name):
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")

canonical_by_norm = {normalize(c): c for c in CLASSES}
class_dirs = {}
for root, dirs, files in os.walk(DATA_ROOT):
    norm_root_name = normalize(os.path.basename(root))
    if norm_root_name in canonical_by_norm and any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
        class_dirs.setdefault(canonical_by_norm[norm_root_name], []).append(root)
assert all(c in class_dirs for c in CLASSES), "Could not find all 4 class folders - check the dataset input."

records = []
for cls, dirs_for_cls in class_dirs.items():
    for cls_dir in dirs_for_cls:
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                records.append({"filepath": os.path.join(cls_dir, fname), "label": cls})
df = pd.DataFrame(records).drop_duplicates(subset="filepath").reset_index(drop=True)
print("Total images:", len(df))
print(df["label"].value_counts().to_string())


In [ ]:
# 3. Identical fixed split (SPLIT_SEED=42) -> directly comparable with Experiments 1-2
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SPLIT_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SPLIT_SEED)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

from sklearn.utils.class_weight import compute_class_weight
class_labels = sorted(df["label"].unique())
cw = compute_class_weight(class_weight="balanced", classes=np.array(class_labels), y=train_df["label"])
class_weight_dict = {label_to_idx[k]: float(v) for k, v in zip(class_labels, cw)}
print("Class weights:", class_weight_dict)


In [ ]:
# 4. Data pipeline. Augmentation lives INSIDE the model (Keras layers are active only during training),
# so validation/test pipelines are plain and deterministic.
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32)

def make_dataset(frame, training=False, seed=42, flip=False):
    paths = frame["filepath"].values
    labels = np.array([label_to_idx[l] for l in frame["label"].values])
    labels_oh = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES)).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_oh))
    ds = ds.map(lambda p, l: (load_image(p), l), num_parallel_calls=AUTOTUNE)
    if flip:
        ds = ds.map(lambda x, y: (tf.image.flip_left_right(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ], name="augmentation")


In [ ]:
# 5. Model builders: one shared head design for every backbone, so the comparison is about the backbone only
from tensorflow.keras.applications import ResNet50, ConvNeXtTiny, EfficientNetV2S

def build_model(arch):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = build_augmentation()(inputs)
    if arch == "resnet50":
        x = tf.keras.applications.resnet50.preprocess_input(x)
        base = ResNet50(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    elif arch == "convnext_tiny":      # preprocessing is built into the Keras ConvNeXt (expects 0-255 input)
        base = ConvNeXtTiny(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    elif arch == "efficientnetv2_s":   # preprocessing is built into Keras EfficientNetV2 (expects 0-255 input)
        base = EfficientNetV2S(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    else:
        raise ValueError(arch)
    feat = layers.GlobalAveragePooling2D(name="gap")(base.output)
    h = layers.Dropout(0.3)(feat)
    h = layers.Dense(256, activation="relu")(h)
    h = layers.Dropout(0.3)(h)
    outputs = layers.Dense(len(CLASSES), activation="softmax", dtype="float32", name="predictions")(h)
    model = models.Model(inputs, outputs, name=arch)
    for layer in base.layers:
        layer.trainable = False
    return model, base

def unfreeze_all_but_bn(base):
    for layer in base.layers:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

def make_loss():
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

def make_phase2_optimizer(steps_total):
    # v2 fix: the first pilot collapsed at peak LR 1e-4 (model predicted a single class). 2e-5 + gradient clipping is stable
    # with the heavy ModerateDemented class weight (24.9) used since Experiments 1-2.
    schedule = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=2e-5, decay_steps=max(steps_total, 1), alpha=0.05)
    try:
        return optimizers.AdamW(learning_rate=schedule, weight_decay=1e-4, clipnorm=1.0)
    except Exception:
        return optimizers.Adam(learning_rate=schedule, clipnorm=1.0)

def train_two_phase(model, base, train_ds, val_ds):
    # Phase 1: head only
    model.compile(optimizer=optimizers.Adam(1e-3, clipnorm=1.0), loss=make_loss(), metrics=["accuracy"])
    cb1 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=0)]
    model.fit(train_ds, validation_data=val_ds, epochs=P1_EPOCHS, class_weight=class_weight_dict, callbacks=cb1, verbose=2)
    # Phase 2: fine-tune everything except BatchNorm, AdamW + cosine decay
    unfreeze_all_but_bn(base)
    steps_total = int(np.ceil(len(train_df) / BATCH_SIZE)) * P2_EPOCHS
    model.compile(optimizer=make_phase2_optimizer(steps_total), loss=make_loss(), metrics=["accuracy"])
    cb2 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=0),
           callbacks.TerminateOnNaN()]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=P2_EPOCHS, class_weight=class_weight_dict, callbacks=cb2, verbose=2)
    best_val = float(max(hist.history["val_accuracy"]))
    return model, best_val

def predict_probs(model, ds):
    probs = model.predict(ds, verbose=0)
    return np.asarray(probs, dtype="float64")

def labels_of(frame):
    return np.array([label_to_idx[l] for l in frame["label"].values])

def metrics_for(y_true, probs):
    y_pred = np.argmax(probs, axis=1)
    y_oh = tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES))
    try:
        auc = float(roc_auc_score(y_oh, probs, average="macro", multi_class="ovr"))
    except Exception:
        auc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro": auc,
    }


In [ ]:
# 6. MAIN LOOP: for each seed and architecture -> train (validation-only selection) -> evaluate once on test (plain + TTA)
# Everything is written to disk as soon as it is computed, and finished (seed, arch) pairs are skipped on re-run.
y_test = labels_of(test_df)
np.save("y_test.npy", y_test)
RESULTS_CSV = "exp3_per_model_results.csv"
rows = pd.read_csv(RESULTS_CSV).to_dict("records") if os.path.exists(RESULTS_CSV) else []

for seed in SEEDS:
    for arch in ARCHS:
        tag = f"{arch}_seed{seed}"
        if os.path.exists(f"probs_tta_{tag}.npy"):
            print(f"[skip] {tag} already done"); continue
        print(f"\n===== {tag} =====")
        t0 = time.time()
        tf.keras.backend.clear_session(); gc.collect()
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

        train_ds = make_dataset(train_df, training=True, seed=seed)
        val_ds = make_dataset(val_df)
        test_ds = make_dataset(test_df)
        test_ds_flip = make_dataset(test_df, flip=True)

        model, base = build_model(arch)
        model, best_val = train_two_phase(model, base, train_ds, val_ds)

        p_plain = predict_probs(model, test_ds)
        p_flip = predict_probs(model, test_ds_flip)
        p_tta = (p_plain + p_flip) / 2.0
        m_plain, m_tta = metrics_for(y_test, p_plain), metrics_for(y_test, p_tta)
        np.save(f"probs_plain_{tag}.npy", p_plain); np.save(f"probs_tta_{tag}.npy", p_tta)

        row = {"arch": arch, "seed": seed, "val_best_accuracy": best_val,
               "test_accuracy_plain": m_plain["accuracy"], "test_accuracy_tta": m_tta["accuracy"],
               "f1_macro_plain": m_plain["f1_macro"], "f1_macro_tta": m_tta["f1_macro"],
               "auc_macro_tta": m_tta["auc_macro"], "minutes": round((time.time() - t0) / 60, 1)}
        rows.append(row); pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
        print(f"  {tag}: val_best={best_val:.4f} | test plain={m_plain['accuracy']:.4f} | test TTA={m_tta['accuracy']:.4f} | {row['minutes']} min")

        del model, base, train_ds, val_ds, test_ds, test_ds_flip
        tf.keras.backend.clear_session(); gc.collect()


In [ ]:
# 7. Aggregate single-model results (mean +/- std over seeds), for each architecture
res = pd.DataFrame(rows)
summary = {}
for arch in ARCHS:
    sub = res[res["arch"] == arch]
    if len(sub) == 0: continue
    summary[arch] = {
        "n_seeds": int(len(sub)),
        "val_best_accuracy": [float(sub["val_best_accuracy"].mean()), float(sub["val_best_accuracy"].std(ddof=1)) if len(sub) > 1 else 0.0],
        "test_accuracy_plain": [float(sub["test_accuracy_plain"].mean()), float(sub["test_accuracy_plain"].std(ddof=1)) if len(sub) > 1 else 0.0],
        "test_accuracy_tta": [float(sub["test_accuracy_tta"].mean()), float(sub["test_accuracy_tta"].std(ddof=1)) if len(sub) > 1 else 0.0],
        "f1_macro_tta": [float(sub["f1_macro_tta"].mean()), float(sub["f1_macro_tta"].std(ddof=1)) if len(sub) > 1 else 0.0],
    }
print("Single-model results (mean, std over seeds):")
for arch, s in summary.items():
    print(f"  {arch:18s} n={s['n_seeds']}  plain acc {100*s['test_accuracy_plain'][0]:.2f} +/- {100*s['test_accuracy_plain'][1]:.2f}"
          f" | TTA acc {100*s['test_accuracy_tta'][0]:.2f} +/- {100*s['test_accuracy_tta'][1]:.2f}"
          f" | TTA macro-F1 {100*s['f1_macro_tta'][0]:.2f} +/- {100*s['f1_macro_tta'][1]:.2f}")
print("\nReference from Experiments 1-2: fusion 90.56 +/- 2.21 | ResNet50-only (fair) 91.35 +/- 1.15 | fusion 5-seed ensemble 91.98")
json.dump(summary, open("exp3_single_model_summary.json", "w"), indent=2)


In [ ]:
# 8. Ensembles (soft voting over saved probabilities; no test feedback is used to choose members -
# every trained model is included, nothing is cherry-picked).
import glob
def load_probs(arch=None):
    out = []
    for f in sorted(glob.glob("probs_tta_*.npy")):
        if arch is None or os.path.basename(f).startswith(f"probs_tta_{arch}_"):
            out.append(np.load(f))
    return out

def hard_vote(prob_list):
    stack = np.stack(prob_list, axis=0); soft = stack.mean(axis=0)
    votes = stack.argmax(axis=2); preds = []
    for i in range(votes.shape[1]):
        counts = np.bincount(votes[:, i], minlength=len(CLASSES))
        top = np.flatnonzero(counts == counts.max())
        preds.append(top[0] if len(top) == 1 else top[np.argmax(soft[i, top])])
    return np.array(preds)

ens_rows = []
def add_ens(name, plist):
    soft = np.mean(np.stack(plist, axis=0), axis=0)
    m = metrics_for(y_test, soft)
    ens_rows.append({"ensemble": name, "n_models": len(plist), "vote": "soft", **m})
    hv = hard_vote(plist)
    ens_rows.append({"ensemble": name, "n_models": len(plist), "vote": "hard",
                     "accuracy": float(accuracy_score(y_test, hv)),
                     "precision_macro": float(precision_score(y_test, hv, average="macro", zero_division=0)),
                     "recall_macro": float(recall_score(y_test, hv, average="macro", zero_division=0)),
                     "f1_macro": float(f1_score(y_test, hv, average="macro", zero_division=0)), "auc_macro": float("nan")})
    return soft

for arch in ARCHS:
    pl = load_probs(arch)
    if pl: add_ens(f"{arch} seeds-only", pl)
all_p = load_probs(None)
diverse_soft = add_ens("diverse (all architectures, all seeds)", all_p)

ens_df = pd.DataFrame(ens_rows)
ens_df.to_csv("exp3_ensemble_results.csv", index=False)
print(ens_df.assign(accuracy=lambda d: (100*d["accuracy"]).round(2), f1_macro=lambda d: (100*d["f1_macro"]).round(2),
                    precision_macro=lambda d: (100*d["precision_macro"]).round(2),
                    recall_macro=lambda d: (100*d["recall_macro"]).round(2)).to_string(index=False))
print("\nReference: Experiments 1-2 fusion 5-seed soft ensemble = 91.98% (hard 92.08%).")


In [ ]:
# 9. Per-class report and confusion matrix for the diverse ensemble; save figure + JSON
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = np.argmax(diverse_soft, axis=1)
print(classification_report(y_test, pred, target_names=CLASSES, digits=4, zero_division=0))
cm_ = confusion_matrix(y_test, pred, labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm_, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Diverse ensemble - test confusion matrix")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm_[i, j]), ha="center", va="center", color="white" if cm_[i, j] > cm_.max()/2 else "black")
plt.tight_layout(); plt.savefig("exp3_diverse_ensemble_confusion_matrix.png", dpi=200); plt.close()

report = classification_report(y_test, pred, target_names=CLASSES, digits=4, zero_division=0, output_dict=True)
json.dump({"per_class_report": report, "confusion_matrix": cm_.tolist()}, open("exp3_diverse_ensemble_per_class.json", "w"), indent=2)
print("Saved exp3_diverse_ensemble_confusion_matrix.png and exp3_diverse_ensemble_per_class.json")


## What to download and send back

From the notebook's **Output** tab (after the commit finishes):

- `exp3_per_model_results.csv` - one row per (architecture, seed)
- `exp3_single_model_summary.json` - mean +/- std per architecture
- `exp3_ensemble_results.csv` - per-architecture and diverse ensembles (soft and hard vote)
- `exp3_diverse_ensemble_per_class.json` and `exp3_diverse_ensemble_confusion_matrix.png`

**How the results will be reported.** All numbers go into the paper as-is, with the same multi-seed discipline. Two things are checked first:
(1) the gap between *validation-best* and *test* accuracy (a large gap would signal over-selection), and
(2) whether any score above about 96% appears - on this dataset that would be a warning sign of slice-level leakage between train and test
(many slices come from the same patient and the dataset has no patient IDs), not automatically a success.